# 02 — Retrieval Sanity Checks

Verifies FR-C1..C4: domain filtering works, top-k is configurable, sources are
preserved for citations, and **Arabic questions retrieve Arabic documents**
even though English near-matches exist in the same corpus.

Run `python -m src.vectorstore` first — this notebook needs a built index.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src.retriever import retrieve, retrieve_multi_domain, unique_sources
from src.settings import settings

print("top-k =", settings.retrieval_top_k)

## Cross-language retrieval (FR-C4)

Queries taken from `tests/eval_cases.jsonl` — the Arabic ones must surface
Arabic or bilingual sources.

In [ ]:
import json

cases = [
    json.loads(l)
    for l in settings.eval_cases_path.read_text(encoding="utf-8").splitlines()
    if l.strip()
]

for case in cases:
    if case["id"] not in {"EVAL-002", "EVAL-007", "EVAL-009"}:
        continue
    hits = retrieve_multi_domain(case["input"], list(settings.domains))
    print(f"\n### {case['id']} ({case['language']}) domain={case['expected_domain']}")
    print("Q:", case["input"][:100])
    print("expected sources:", case["expected_sources"])
    for h in hits[:5]:
        marker = "✓" if h.source_filename in case["expected_sources"] else " "
        print(f"  {marker} {h.score:.3f} [{h.domain}/{h.language}] {h.source_filename}")

## Domain filtering (FR-C1)

The same question retrieved with each single-domain filter should only
return chunks tagged with that domain.

In [ ]:
q = "How many vacation days does a new hire get?"
for domain in settings.domains:
    hits = retrieve(q, domain, top_k=3)
    domains = {h.domain for h in hits}
    assert domains <= {domain}, f"filter leaked: {domains}"
    print(f"{domain:8s} -> {[h.source_filename for h in hits]}")
print("\nDomain filter: OK (FR-C1)")

## Multi-document questions

EVAL-007 expects **both** FIN-001 and FIN-002 — single-document retrieval
loses Contextual Recall points. Check the fan-out retrieval covers them.

In [ ]:
case = next(c for c in cases if c["id"] == "EVAL-007")
hits = retrieve_multi_domain(case["input"], list(settings.domains), top_k=8)
retrieved = set(unique_sources(hits))
expected = set(case["expected_sources"])
print("expected:", sorted(expected))
print("retrieved:", sorted(retrieved))
print("missing:", sorted(expected - retrieved) or "none — full coverage ✓")